# Spanning: whether either factor set prices what the other does

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.factors.spanning`

**Modules covered** `factors/spanning.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

Spanning asks whether one factor set adds anything the other does not already price. The entry point runs the test in both directions, with the GRS statistic and its p-value beside the regression form, and reports the second-pass premium estimate with its own caveat.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `factors/spanning.py::directions` traces to the spanning test of Huberman & Kandel (1987), Journal of Finance 42(4), in the regression form that the second factor set adds nothing the first did not price
- `factors/spanning.py::fama_macbeth` traces to the Fama & MacBeth (1973), Journal of Political Economy 81(3), two-pass estimate, reported as unsupported at this panel's size rather than as a result
- `factors/spanning.py::grs` traces to the Gibbons, Ross & Shanken (1989), Econometrica 57(5), statistic reported beside the regression form
- `factors/spanning.py::main` traces to the spanning test of Huberman & Kandel (1987), Journal of Finance 42(4), in the regression form that the second factor set adds nothing the first did not price
- `factors/spanning.py::premiums` traces to the second-pass premium estimate and the errors-in-variables caveat that makes it a descriptive reading on this panel rather than a test

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`factors/spanning.py`**

Spanning, tested in both directions, and the premia the factors are credited with.

The evidential core of the factor work is not which set explains more variance. It is whether
one set is needed at all once the other is present. So the two sets are tested against each
other: the **named set as benchmark with the statistical components as test assets** is the
headline, because it asks whether the published and constructed factors describe this panel,
and the reverse direction prints beside it rather than being dropped, because a one-directional
answer to a two-directional question is a selection of the result.

The test form is Huberman and Kandel (1987) in the spanning reading of their 1987 result: regress
each test asset
on the benchmark set, and the benchmark spans the test assets when, jointly, **all intercepts
are zero** and **the rows of the loadings sum to one**. Gibbons, Ross and Shanken (1989)
supplies the finite-sample F test for the joint-intercepts half. Both halves are reported.

Two limits travel with the result and neither is a footnote. The rows-sum-to-one half is a
statement about a *fully invested* benchmark: it says that a portfolio of the benchmark factors
reproduces the test asset with no residual, and one condition for that is that the exposures add
up to the whole asset. This benchmark mixes an excess-market series with zero-cost spread series
(the momentum, term, credit and high-yield legs invest nothing), so the condition is reported
as computed and its failure is not read as a spanning verdict on its own; the GRS half carries
the verdict. And **GRS assumes iid normal residuals**, which monthly ETF returns violate in the
tails, so the statistic travels with the assumption rather than resting on it silently.

Nothing here is evidence about European UCITS ETFs in general, and no published head-to-head of
statistical against fundamental risk models on this universe exists to replicate. The result is
a statement about these eleven sleeves over this window.

## 3. The data contract it consumes, and the as-of rule

The test assets are the sleeves, the factors are the named set, and the regression runs on the months both cover at the count the retention rule holds. Both directions are run on the same month set, so the two p-values are comparable. The premium estimate is descriptive: with eleven series and one out-of-sample window the second pass is underpowered, and the layer says so rather than reporting a t-statistic as a result.

## 4. The worked example on small numbers, with the identity checked

Two planted cases show what the statistic reads. Test assets built from the factors plus independent noise leave the alphas at zero and the test keeps its size; the same assets given a constant 40 bp a month the factors do not carry are rejected outright. Assets exactly reproducible from the factors are *not* the case to check: both the alphas and their residual covariance sit at rounding level there, and the ratio of the two is numerical noise rather than a statistic.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import numpy as np
import pandas as pd

from portfolio_workbench.factors import spanning

months = pd.period_range("2020-01", periods=60, freq="M")
rng = np.random.default_rng(11)
factors = pd.DataFrame(
    {"market": rng.normal(0.004, 0.03, 60), "credit": rng.normal(0.001, 0.01, 60)}, index=months
)
noise = rng.normal(0.0, 0.004, (60, 2))
spanned = pd.DataFrame(
    {
        "a": 0.9 * factors["market"] + 0.4 * factors["credit"] + noise[:, 0],
        "b": 1.1 * factors["market"] - 0.2 * factors["credit"] + noise[:, 1],
    },
    index=months,
)

# Under the null the test keeps its size: this seed does not reject.
assert spanning.grs(spanned, factors)["p_value"] > 0.05

# A constant 40 bp a month the factors do not carry is what the test exists to detect.
planted = spanned + 0.004
assert spanning.grs(planted, factors)["p_value"] < 0.05
print(f"null case p {spanning.grs(spanned, factors)['p_value']:.4f}, planted alpha p "
      f"{spanning.grs(planted, factors)['p_value']:.4f}")

null case p 0.8783, planted alpha p 0.0000


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.factors import spanning

print("the test form: Huberman & Kandel (1987) read as a regression of the test assets on the factors")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
the test form: Huberman & Kandel (1987) read as a regression of the test assets on the factors


In [3]:
import subprocess
import sys
from pathlib import Path

# The package is imported from the repository root, so the run needs the root as its working
# directory rather than wherever the kernel was started. Walked up from the kernel's own directory
# rather than written in at generation time: an absolute path here would name one workstation, and
# the notebook is a file every reader runs on their own.
root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "portfolio_workbench").is_dir()
)

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.factors.spanning"], capture_output=True, text=True, cwd=root
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[factor] spanning over 190 months at 2 component(s), the count the rule retains; iid-normal residuals assumed, and monthly returns violate that in the tails
    headline: the named set (10 factors) against the components (2 assets) - GRS 4.744, p 0.0098, joint intercepts rejected
    reverse: the components (2) against the named set (10 assets) - GRS 2.970, p 0.0018, printed beside the headline rather than dropped
    rows-sum-to-one half, headline direction: the loadings' row sums run +0.34..+1.25 against 1.00 under spanning, largest deviation 0.66; the components are fully invested portfolios, so this is a statement about net exposure
    rows-sum-to-one half, reverse direction: -12.90..+16.94, reported as computed - the test assets there are the named factors themselves, and the zero-cost spread series among them are not the fully invested case the condition is stated for
[factor] spanning over 190 months at 3 component(s), the pre-registered count; iid-normal residuals assumed, and

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The test is run at the retained count and on the months both legs cover, so its power is bounded by that count and by that span: a failure to reject is the expected outcome on a panel of diversified index sleeves and is not evidence that the sets are equivalent. Where the test does reject, the direction matters and both are printed. A reader must not read the second-pass premium estimate as a priced-factor result: at this panel's size the standard error dominates it, and the layer prints the estimate beside that statement rather than instead of it.

## 7. What this module does not establish

Nothing here establishes that a spanning rejection is economically meaningful, or that the two sets are the same model. The test is about whether one set's information is contained in the other's on this sample, and the answer is sample-specific.